# Sentiment Classifier Training — Preprocessed SemEval 2017 / tweet_eval

Fine-tunes **DistilBERT** (Sanh et al., 2019) on the **preprocessed** SemEval 2017 Tweet Sentiment corpus.  
Training data is loaded from locally preprocessed CSV files (not directly from HuggingFace).

**Preprocessing applied** (see `PREPROCESSING.md` for full details):
- HTML entities decoded
- URLs replaced with `[URL]`
- `@mentions` replaced with `[USER]`
- `#` stripped from hashtags — word kept
- Repeated punctuation normalised (`!!!` → `!`)
- Extra whitespace collapsed
- Rows with fewer than 3 words removed

**Dataset sizes after preprocessing:**
- Train : 45,613 rows
- Val   : 2,000 rows
- Test  : 12,241 rows

**Hyperparameters are identical to the baseline** (raw data run) for a fair comparison.

### Steps before running
1. `Runtime → Change runtime type → T4 GPU` → Save
2. Run **Cell 1 only** first — installs packages and restarts kernel automatically
3. After restart, run **all remaining cells** from Cell 2 onward

In [ ]:
# ── CELL 1 — Install packages  (run this cell first, alone) ───────────────────
!pip install "datasets==3.2.0" "transformers>=4.40.0" accelerate scikit-learn pandas -q
print('Packages installed.')
print('Restarting kernel automatically...')

import os
os.kill(os.getpid(), 9)

In [ ]:
# ── CELL 2 — Check GPU  (run after kernel restarts) ───────────────────────────
import torch
print('GPU available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device        :', torch.cuda.get_device_name(0))
    print('Memory (GB)   :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU')

import datasets, transformers
print('datasets version    :', datasets.__version__)
print('transformers version:', transformers.__version__)

In [ ]:
# ── CELL 3 — Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_PATH = '/content/drive/MyDrive/sentiment_classifier_preprocessed'
os.makedirs(SAVE_PATH, exist_ok=True)
print('Model will be saved to:', SAVE_PATH)

In [ ]:
# ── CELL 4 — Upload preprocessed CSV files ────────────────────────────────────
# Upload these 3 files from sentiment_preprocessed_data_set/ on your local machine:
#   tweet_eval_train_clean.csv
#   tweet_eval_val_clean.csv
#   tweet_eval_test_clean.csv

from google.colab import files
import shutil, os

os.makedirs('data', exist_ok=True)

print('Upload: tweet_eval_train_clean.csv, tweet_eval_val_clean.csv, tweet_eval_test_clean.csv')
uploaded = files.upload()

for fname in uploaded:
    shutil.move(fname, f'data/{fname}')
    print(f'  Moved {fname} -> data/{fname}')

print('\nFiles in data/:', os.listdir('data/'))

In [ ]:
# ── CELL 5 — Load CSVs and verify label distributions ─────────────────────────
import pandas as pd
from collections import Counter

ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}

df_train = pd.read_csv('data/tweet_eval_train_clean.csv')
df_val   = pd.read_csv('data/tweet_eval_val_clean.csv')
df_test  = pd.read_csv('data/tweet_eval_test_clean.csv')

print('=== Dataset sizes ===')
print(f'  Train : {len(df_train):,}')
print(f'  Val   : {len(df_val):,}')
print(f'  Test  : {len(df_test):,}')

for name, df in [('TRAIN', df_train), ('VAL', df_val), ('TEST', df_test)]:
    counts = Counter(df['label_name'].tolist())
    print(f'\n{name} label distribution:')
    for lid, lname in ID2LABEL.items():
        pct = counts[lname] / len(df) * 100
        print(f'  {lname:10s}: {counts[lname]:,}  ({pct:.1f}%)')

print('\nSample cleaned rows:')
print(df_train[['text', 'label_name']].head(5).to_string())

In [ ]:
# ── CELL 6 — Convert to HuggingFace Dataset and tokenise ──────────────────────
from datasets import Dataset
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def df_to_hf(df):
    return Dataset.from_dict({
        'text':  df['text'].tolist(),
        'label': df['label'].tolist(),
    })

raw = {
    'train':      df_to_hf(df_train),
    'validation': df_to_hf(df_val),
    'test':       df_to_hf(df_test),
}

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=128,
        padding='max_length',
    )

tokenised = {}
for split, ds in raw.items():
    tok = ds.map(tokenize, batched=True, batch_size=512)
    tok = tok.rename_column('label', 'labels')
    tok.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    tokenised[split] = tok

print('Tokenisation complete.')
print('Input shape (first train sample):', tokenised['train'][0]['input_ids'].shape)

In [ ]:
# ── CELL 7 — Load DistilBERT with 3-class classification head ─────────────────
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total:,}')
print(f'Trainable parameters: {trainable:,}')

In [ ]:
# ── CELL 8 — Train with class weights + early stopping  (~15-20 min on T4) ────
# Changes vs baseline:
#   - WeightedTrainer: upweights negative class (15.6% of data) in the loss
#   - 6 epochs max with early stopping patience=2 on val f1_macro
#   - Warmup updated to 10% of new total steps
import numpy as np
import torch
import torch.nn as nn
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import f1_score, accuracy_score

# ── Class weights (inverse-frequency) ─────────────────────────────────────────
# negative: 7,093 rows  weight = 45613 / (3 * 7093)  = 2.143
# neutral : 20,671 rows weight = 45613 / (3 * 20671) = 0.736
# positive: 17,849 rows weight = 45613 / (3 * 17849) = 0.852
N_TRAIN    = len(df_train)
N_CLASSES  = 3
counts     = [df_train[df_train['label'] == i].shape[0] for i in range(N_CLASSES)]
weights    = torch.tensor(
    [N_TRAIN / (N_CLASSES * c) for c in counts], dtype=torch.float
)
print('Class weights:', [round(w.item(), 3) for w in weights])
print('  negative:', round(weights[0].item(), 3),
      '  neutral:', round(weights[1].item(), 3),
      '  positive:', round(weights[2].item(), 3))


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(weight=weights.to(labels.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1_per_class = f1_score(labels, preds, average=None, labels=[0, 1, 2])
    return {
        'accuracy'    : round(accuracy_score(labels, preds), 4),
        'f1_macro'    : round(f1_score(labels, preds, average='macro'), 4),
        'f1_negative' : round(float(f1_per_class[0]), 4),
        'f1_neutral'  : round(float(f1_per_class[1]), 4),
        'f1_positive' : round(float(f1_per_class[2]), 4),
    }


# warmup = 10% of total steps  (45613 / 64) * 6 epochs ≈ 4278 steps
WARMUP_STEPS = 428
NUM_EPOCHS   = 6

training_args = TrainingArguments(
    output_dir='/content/checkpoints',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenised['train'],
    eval_dataset=tokenised['validation'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f'Starting training — {NUM_EPOCHS} epochs max, early stopping patience=2...')
trainer.train()

In [ ]:
# ── CELL 9 — Evaluate on held-out test set ────────────────────────────────────
test_results = trainer.evaluate(tokenised['test'])

print('=' * 55)
print('  TEST SET RESULTS  —  preprocessed model')
print('=' * 55)
print(f"  Accuracy    : {test_results.get('eval_accuracy', 0):.4f}")
print(f"  F1 Macro    : {test_results.get('eval_f1_macro', 0):.4f}")
print(f"  F1 Negative : {test_results.get('eval_f1_negative', 0):.4f}")
print(f"  F1 Neutral  : {test_results.get('eval_f1_neutral', 0):.4f}")
print(f"  F1 Positive : {test_results.get('eval_f1_positive', 0):.4f}")
print('=' * 55)

In [ ]:
# ── CELL 10 — Save model + tokeniser to Google Drive ──────────────────────────
import json

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

metrics = {
    'base_model'       : 'distilbert-base-uncased',
    'dataset'          : 'cardiffnlp/tweet_eval (sentiment, SemEval 2017) — preprocessed',
    'preprocessing'    : [
        'HTML entities decoded',
        'URLs replaced with [URL]',
        '@mentions replaced with [USER]',
        '# stripped from hashtags',
        'Repeated punctuation normalised (!!!->!)',
        'Extra whitespace collapsed',
        'Rows < 3 words removed',
    ],
    'class_weights'    : {'negative': round(weights[0].item(), 3),
                          'neutral':  round(weights[1].item(), 3),
                          'positive': round(weights[2].item(), 3)},
    'train_size'       : len(df_train),
    'val_size'         : len(df_val),
    'test_size'        : len(df_test),
    'max_epochs'       : NUM_EPOCHS,
    'early_stopping_patience': 2,
    'learning_rate'    : 2e-5,
    'warmup_steps'     : WARMUP_STEPS,
    'test_results'     : {k.replace('eval_', ''): v for k, v in test_results.items()},
}
with open(f'{SAVE_PATH}/training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved to Google Drive:', SAVE_PATH)
print('Files:')
for fname in sorted(os.listdir(SAVE_PATH)):
    size = os.path.getsize(f'{SAVE_PATH}/{fname}')
    print(f'  {fname:40s}  {size/1e6:.1f} MB')

In [ ]:
# ── CELL 11 — Download model as ZIP ──────────────────────────────────────────
import shutil
from google.colab import files

zip_path = '/content/sentiment_classifier_preprocessed'
shutil.make_archive(zip_path, 'zip', SAVE_PATH)

zip_size = os.path.getsize(f'{zip_path}.zip') / 1e6
print(f'ZIP size: {zip_size:.0f} MB')
print('Starting download...')
files.download(f'{zip_path}.zip')

## After downloading

1. Unzip `sentiment_classifier_preprocessed.zip`
2. Compare `training_metrics.json` against the baseline model's `training_metrics.json`
3. If preprocessed model scores are higher → place at `m3_implementation/memory/models/sentiment_classifier/` (replacing baseline)
4. If scores are similar or lower → keep the baseline model

`training_metrics.json` contains accuracy, macro F1, and per-class F1 for direct comparison.

---
## Evaluation Analysis (run independently — no retraining needed)
The cells below load your **already-trained preprocessed model** from Google Drive and generate full evaluation visualisations.  
Run from Cell E1 onward — no need to re-run training cells.

In [ ]:
# ── CELL E1 — Setup: mount Drive + install packages ───────────────────────────
!pip install "datasets==3.2.0" "transformers>=4.40.0" scikit-learn seaborn pandas -q

from google.colab import drive
drive.mount('/content/drive')

import os
MODEL_PATH = '/content/drive/MyDrive/sentiment_classifier_preprocessed'
EVAL_PATH  = '/content/drive/MyDrive/sentiment_classifier_preprocessed/eval_results'
os.makedirs(EVAL_PATH, exist_ok=True)

print('Model path :', MODEL_PATH)
print('Eval output:', EVAL_PATH)
for f in ['config.json', 'model.safetensors', 'tokenizer.json']:
    status = '✓' if os.path.isfile(f'{MODEL_PATH}/{f}') else '✗ MISSING'
    print(f'  {f}: {status}')

In [ ]:
# ── CELL E2 — Upload test CSV + load model, run predictions ───────────────────
import numpy as np
import torch
import pandas as pd
from google.colab import files
import shutil
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

ID2LABEL = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

# Upload tweet_eval_test_clean.csv
os.makedirs('data', exist_ok=True)
print('Upload: tweet_eval_test_clean.csv')
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f'data/{fname}')

df_test = pd.read_csv('data/tweet_eval_test_clean.csv')
print(f'Test set loaded: {len(df_test):,} rows')

# Load trained model
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_PATH)
model     = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f'Model loaded — running on {device}')

# Batch inference
BATCH = 128
texts  = df_test['text'].tolist()
labels = df_test['label'].tolist()
all_preds = []

for i in range(0, len(texts), BATCH):
    enc = tokenizer(texts[i:i+BATCH], truncation=True, max_length=128,
                    padding='max_length', return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())

y_pred = np.array(all_preds)
y_true = np.array(labels)
print(f'Inference complete — {len(y_pred):,} predictions')

In [ ]:
# ── CELL E3 — VISUALISATION 1: Confusion Matrix ───────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

label_names = ['Negative', 'Neutral', 'Positive']
cm     = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f'Confusion Matrix — Preprocessed DistilBERT on SemEval 2017 Test Set ({len(y_true):,} samples)',
    fontsize=13, fontweight='bold'
)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=axes[0],
            linewidths=0.5, linecolor='gray', annot_kws={'size': 13})
axes[0].set_title('Raw Counts', fontsize=12)
axes[0].set_xlabel('Predicted', fontsize=11)
axes[0].set_ylabel('True', fontsize=11)

sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=axes[1],
            linewidths=0.5, linecolor='gray', annot_kws={'size': 13})
axes[1].set_title('Row-normalised (%)', fontsize=12)
axes[1].set_xlabel('Predicted', fontsize=11)
axes[1].set_ylabel('True', fontsize=11)

plt.tight_layout()
plt.savefig(f'{EVAL_PATH}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

In [ ]:
# ── CELL E4 — VISUALISATION 2: Per-class Precision / Recall / F1 ──────────────
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report

label_names = ['Negative', 'Neutral', 'Positive']
report = classification_report(y_true, y_pred, target_names=label_names, output_dict=True)

metrics_list = ['precision', 'recall', 'f1-score']
x      = np.arange(len(label_names))
width  = 0.25
colors = ['#4C72B0', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(10, 6))
for i, (metric, color) in enumerate(zip(metrics_list, colors)):
    values = [report[label][metric] for label in label_names]
    bars = ax.bar(x + i * width, values, width, label=metric.capitalize(),
                  color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

macro_f1 = report['macro avg']['f1-score']
ax.axhline(y=macro_f1, color='black', linestyle='--', linewidth=1.5,
           label=f'Macro F1 = {macro_f1:.4f}')

ax.set_xlabel('Sentiment Class', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Per-class Metrics — Preprocessed DistilBERT on SemEval 2017 Test Set',
             fontsize=12, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(label_names, fontsize=12)
ax.set_ylim(0, 1.10)
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(f'{EVAL_PATH}/per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFull Classification Report:')
print(classification_report(y_true, y_pred, target_names=label_names))
print('Saved: per_class_metrics.png')

In [ ]:
# ── CELL E5 — VISUALISATION 3: Metrics Summary Card ──────────────────────────
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
import json

acc      = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro')
f1_neg   = f1_score(y_true, y_pred, average=None)[0]
f1_neu   = f1_score(y_true, y_pred, average=None)[1]
f1_pos   = f1_score(y_true, y_pred, average=None)[2]

fig, ax = plt.subplots(figsize=(8, 5))
ax.axis('off')
fig.patch.set_facecolor('#f8f9fa')

ax.text(0.5, 0.95, 'DistilBERT Sentiment Classifier — Preprocessed Data Evaluation',
        ha='center', va='top', fontsize=13, fontweight='bold', transform=ax.transAxes)
ax.text(0.5, 0.87,
        f'Dataset: SemEval 2017 (preprocessed)  |  Test set: {len(y_true):,} samples',
        ha='center', va='top', fontsize=10, color='gray', transform=ax.transAxes)

rows = [
    ('Accuracy',      f'{acc:.4f}',      f'{acc*100:.2f}%'),
    ('Macro F1',      f'{f1_macro:.4f}', f'{f1_macro*100:.2f}%'),
    ('F1 — Negative', f'{f1_neg:.4f}',   ''),
    ('F1 — Neutral',  f'{f1_neu:.4f}',   ''),
    ('F1 — Positive', f'{f1_pos:.4f}',   ''),
]

table = ax.table(cellText=[[r[0], r[1], r[2]] for r in rows],
                 colLabels=['Metric', 'Score', ''],
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 2.0)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#343a40')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#e9ecef')
    cell.set_edgecolor('white')

plt.tight_layout()
plt.savefig(f'{EVAL_PATH}/metrics_summary.png', dpi=150, bbox_inches='tight')
plt.show()

eval_summary = {
    'model':        'distilbert-base-uncased (fine-tuned, preprocessed data)',
    'dataset':      'cardiffnlp/tweet_eval sentiment (SemEval 2017) — preprocessed',
    'test_samples': int(len(y_true)),
    'accuracy':     round(float(acc),      4),
    'f1_macro':     round(float(f1_macro), 4),
    'f1_negative':  round(float(f1_neg),   4),
    'f1_neutral':   round(float(f1_neu),   4),
    'f1_positive':  round(float(f1_pos),   4),
}
with open(f'{EVAL_PATH}/eval_summary.json', 'w') as f:
    json.dump(eval_summary, f, indent=2)

print('Saved: metrics_summary.png')
print('Saved: eval_summary.json')
print('\nEvaluation files in:', EVAL_PATH)